In [ ]:
!pip install -U langchain-google-genai
!pip install langchain langchain-tavily
!pip install langchain-mcp-adapters

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.chat_models import init_chat_model
from google.colab import userdata

In [ ]:
google_api_key = userdata.get('gemini_api_key')
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)

In [ ]:
import requests
from langchain.tools import tool
from google.colab import userdata

In [ ]:
@tool
def search_jobs(skill: str, location: str) -> list:
  """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
  print(f"\nCalling search_jobs tool")
  print(f"Searching jobs for: {skill} in {location}")

  rapidapi_key = userdata.get('rapid_apikey')

  url = "https://jsearch.p.rapidapi.com/search"
  headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
  }
  querystring = {
    "query": f"{skill} in {location}",
    "page": "1",
    "country": "in",
    "employment_types": "INTERN,FULLTIME",
    "job_requirements": "no_experience,under_3_years_experience"
  }
  response = requests.get(url, headers=headers, params=querystring)
  data = response.json()
  jobs = data.get("data", [])
  print(f"Found {len(jobs)} jobs\n")

  result = []
  for job in jobs:
    result.append({
      "title": job.get("job_title"),
      "company": job.get("employer_name"),
      "location": job.get("job_city"),
      "apply_link": job.get("job_apply_link")
    })
  return result


In [ ]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- search tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""


In [ ]:
tavily_api_key = userdata.get('tavily_apikey')

Configuring the MCP Client

In [ ]:
client = MultiServerMCPClient(
    {
        "mcp_tavily": {
            "transport": "http",
            "url": f"https://mcp.tavily.com/mcp/?tavilyApiKey={tavily_api_key}",
        }
    }
)

Executing the Agent using ainvoke()

In [ ]:
from langchain.agents import create_agent

async def skill_map_agent():

  mcp_tools = await client.get_tools()

  # Combine MCP tools with our custom tools
  all_tools = mcp_tools+ [search_jobs]

  agent = create_agent(
    model=model,
  tools=all_tools,
  system_prompt=system_prompt,
  debug=True

  )
  user_query = "What's the demand for generative ai in the industry "

  response = await agent.ainvoke({
    "messages": [{"role": "user", "content": user_query}]
  })
  print(response["messages"][-1].content[0]["text"])


await skill_map_agent()
